# Postprocessing

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import h5py
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
import os
import myfunctions
import h5py
import TM26_unet as unet


GPU = True
MULTIGPU = True

DELETE_OLD_FILES = False
DELETE_OLD_FILES = False

SMALL_SIZE = 18
MEDIUM_SIZE = 22
BIGGER_SIZE = 24
GPU = True

diffLimDir = "dataInputs/"
testDataDir = "output_prep/"
outputDir = "output_post/"
inputDir = "output_infer/"

datasets_name = ["lightfield", "WFSxy", "WFSxy_DM", "WFSxy_DM_intensity", "WFSxy_intensity"]
dataset_name_readable = ["lightfield", "WFSxy", "WFSxy and DM commands", "WFSxy, SHWFS intensity map and DM commands", "WFSxy and SHWFS intensity map"]

batch_size = 64

dataset_numbers = len(datasets_name)



SMALL_SIZE = 18
MEDIUM_SIZE = 22
BIGGER_SIZE = 24
GPU = True



In [ ]:
# for dataset_number in range(dataset_numbers):
dataset_number = 1

## load data

In [ ]:
# with h5py.File(inputDir+datasets_name[dataset_number]+"/predictions.h5", 'r') as temp:
#     inference = temp[list(temp.keys())[0]][()]
# truth = []
# with h5py.File(testDataDir+"ao_ipsf_test_1.h5", 'r') as temp:
#     key = list(temp.keys())
#     tempdata = temp[key[0]][()][:]
#     for i in range(tempdata.shape[0]):
#         truth.append(myfunctions.pixel_adder(tempdata[i],final_shape=[270, 270]))

In [ ]:
with h5py.File(inputDir+datasets_name[dataset_number]+"/predictions.h5", 'r') as temp:
    inference = temp[list(temp.keys())[0]][()]
    
with h5py.File(testDataDir+"ao_ipsf_test_1.h5", 'r') as temp:
    key = list(temp.keys())
    truth = temp[key[0]][()][:]

plt.close('all')
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(inference[0]/np.max(inference[0]))
axes[0].set_title('Normalised inference')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(truth[0]/np.max(truth[0]))
axes[1].set_title('Normalised truth')
fig.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(inference[0]/np.max(inference[0]) - truth[0]/np.max(truth[0]))
axes[2].set_title('Difference (Inference - Truth)')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## FFT

selon mathft.pro

In [ ]:
with h5py.File("dataInputs/ipsf_difflim.h5", 'r') as temp:
    ipsf_difflim = temp[list(temp.keys())[0]][()]/2861.3890710476435
plt.close(0)
plt.figure(0)
plt.imshow(ipsf_difflim)

In [ ]:
def diffraction_limited_otf_circular(freq_cpr, wavelength, diameter):

    freq_cpr = np.asarray(freq_cpr, dtype=float)
    fc = diameter / wavelength
    nu = np.abs(freq_cpr) / fc

    otf = np.zeros_like(nu)
    inside = nu <= 1.0

    x = nu[inside]
    otf[inside] = (2/np.pi) * (np.arccos(x) - x * np.sqrt(1 - x**2))

    return otf, fc

In [ ]:
# D = 17.5e-3 # [m]
# wl = 632.8e-9 # [m]
# dimmat_psf =80
# delta_nu = 2*D/(dimmat_psf*wl)  # Angular frequency sampling
# nu_x = np.linspace(-dimmat_psf/2, dimmat_psf/2-1, dimmat_psf) * delta_nu
# NU_X, NU_Y = np.meshgrid(nu_x, nu_x)
# nu = np.sqrt(NU_X**2 + NU_Y**2)
# nu_n = nu * wl / D
# 
# pad = 2
# padded_truth = np.zeros((dimmat_psf*pad,dimmat_psf*pad))
# padded_truth[0:dimmat_psf, 0:dimmat_psf] = truth[10]

# padded_inf = np.zeros((dimmat_psf*pad,dimmat_psf*pad))
# padded_inf[0:dimmat_psf, 0:dimmat_psf] = inference[10]

# OTF_tel = myfunctions.telescope_otf(nu_n)

# OTF_truth = abs(myfunctions.ft2(padded_truth,delta_nu))[dimmat_psf*pad//2-dimmat_psf//2:dimmat_psf*pad//2+dimmat_psf//2, dimmat_psf*pad//2-dimmat_psf//2:dimmat_psf*pad//2+dimmat_psf//2]/np.sum(truth[10])
# OTF_truth = OTF_truth/np.max(OTF_truth)

# OTF_inf = abs(myfunctions.ft2(padded_inf,delta_nu))[dimmat_psf*pad//2-dimmat_psf//2:dimmat_psf*pad//2+dimmat_psf//2, dimmat_psf*pad//2-dimmat_psf//2:dimmat_psf*pad//2+dimmat_psf//2]/np.sum(inference[10])
# OTF_inf = OTF_inf/np.max(OTF_inf)
# freq = np.linspace(0, D / wl, 40)
# otf, fc = diffraction_limited_otf_circular(freq, wl, D)

# plt.figure(figsize=(6,4))
# # plt.plot(otf, label="Diffraction-limited OTF")
# plt.plot(OTF_tel[40,], label="Diffraction-limited 2d")
# plt.plot(OTF_truth[40,:], label="Truth OTF")
# plt.plot(OTF_inf[40,:], label="Inference OTF")

# # plt.axvline(fc, color="r", linestyle="--", label=f"Cutoff = {fc:.3e} cy/rad")
# plt.xlabel("Angular spatial frequency [cycles/radian]")
# plt.ylabel("OTF / MTF")
# # plt.ylim(0, 1)
# plt.grid(True, alpha=0.3)
# plt.legend()
# plt.tight_layout()
# plt.show()

# # plt.close(0)
# # plt.figure(0)
# # plt.plot(OTF_truth[:,80//2]/np.max(OTF_truth))
# # plt.plot(OTF_inf[:,80//2]/np.max(OTF_inf))
# # plt.plot(OTF_tel[:,80//2])

# Strehl_truth = np.sum(OTF_truth)/np.sum(OTF_tel)
# Strehl_inf = np.sum(OTF_inf)/np.sum(OTF_tel)
# print(Strehl_truth, Strehl_inf, Strehl_truth / Strehl_inf)
# print()

## Strehl



In [ ]:

# relative_strehl_dist = (interpolation_strehl - truth_strehl) / truth_strehl

In [ ]:
lower_bound = truth_strehl * 0.9  # -10%
upper_bound = truth_strehl * 1.1  # +10%

plt.close('all')
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={'height_ratios': [3, 2]})

# Top plot: Strehl ratios
ax.plot(truth_strehl, 'k', label=r'$S_{ref}$')
ax.plot(interpolation_strehl, 'r', label=r'$S_{R}$')
ax.set_ylim([0, 1])
ax.fill_between(range(len(truth_strehl)), lower_bound, upper_bound, 
                color='gray', alpha=0.3, edgecolor='none', label='±10% of expected Strehl ratio')
ax.set_ylabel('Strehl ratio [-]', fontsize=MEDIUM_SIZE)
ax.tick_params(axis='y', labelsize=SMALL_SIZE)
ax.legend(fontsize=SMALL_SIZE)
ax.set_title("Comparison of Strehl ratios for instantaneous PSFs", fontsize=BIGGER_SIZE)
ax.set_xlim([0, inference.shape[0]])
ax.grid()

# Bottom plot: Relative distance (ax2 is now a separate subplot below)
ax2.plot(relative_strehl_dist * 100, 'k', label=r"$\frac{S_{R}-S_{ref}}{S_{ref}}$")
ax2.set_ylabel('Relative distance [%]', fontsize=MEDIUM_SIZE, color='k')
ax2.tick_params(axis='y', labelsize=SMALL_SIZE, colors='k')
symetric_lim = max(abs(np.min(relative_strehl_dist) * 110), abs(np.max(relative_strehl_dist) * 110))
ax2.set_ylim([-symetric_lim, symetric_lim])
ax2.legend(fontsize=MEDIUM_SIZE)
ax2.set_xlabel('Iteration [-]', fontsize=MEDIUM_SIZE)
ax2.set_xlim([0, inference.shape[0]])
ax2.grid()
ax2.tick_params(axis='x', labelsize=SMALL_SIZE)

plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')
plt.figure(figsize=(10, 8))
plt.plot(truth_strehl,interpolation_strehl,'.k')
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('Truth Strehl', fontsize=MEDIUM_SIZE)
plt.ylabel('Interpolation Strehl', fontsize=MEDIUM_SIZE)
plt.xticks(fontsize=SMALL_SIZE)
plt.yticks(fontsize=SMALL_SIZE)
plt.title(r'$S_R(S_{ref})$', fontsize=BIGGER_SIZE)
plt.grid(True)
plt.show()

## Rotational average

1. Find center
2. Average the PSF
3. compare